# Experiment C - Percentage vs Raw Feature Clustering

**Reads**: `../../data/processed/4_activity_contributions.csv`  
**Reads**: `../../data/processed/3_normalized_telemetry.csv`  
**Writes**: `../../experiments/pct_vs_raw_results.csv`

> Prerequisites: Run the core pipeline first (`core/notebooks/01-05`).

---

## Hypothesis

Notebook `05_Clustering` documents a failed early approach: clustering players on their **activity percentages** (`pct_combat`, `pct_collect`, `pct_explore`) instead of normalized raw features.

The claim is that percentages introduce a **Compositional Data Problem** - because they always sum to 1.0, the three axes are linearly dependent. K-Means assumes independent Euclidean axes, so this dependency artificially inflates cluster separation metrics while producing geometrically degenerate clusters.

This experiment reproduces both approaches side-by-side across K=2,3,4,5 to confirm the claim quantitatively.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "../../requirements.txt"])
print("Dependencies OK")

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

PROC = "../../data/processed"
EXP  = "../../experiments"
os.makedirs(EXP, exist_ok=True)

print("Libraries imported.")

## 1. Load Data

In [ ]:
# I need two different pipeline outputs here:
# - notebook 04's output for the percentage features (pct_combat, pct_collect, pct_explore)
# - notebook 03's output for the normalized raw features
# The row counts must match since both describe the same sessions.
path_pct = os.path.join(PROC, "4_activity_contributions.csv")
path_raw = os.path.join(PROC, "3_normalized_telemetry.csv")

assert os.path.exists(path_pct), f"Missing: {path_pct}. Run core pipeline first."
assert os.path.exists(path_raw), f"Missing: {path_raw}. Run core pipeline first."

df_pct = pd.read_csv(path_pct)
df_raw = pd.read_csv(path_raw)

print(f"Loaded {len(df_pct)} rows from 4_activity_contributions.csv")
print(f"Loaded {len(df_raw)} rows from 3_normalized_telemetry.csv")

## 2. Define Feature Sets

**Approach A - Percentages**: The 3 output percentages that sum to 1.0.  
**Approach B - Raw (v2.2)**: The 11 normalized input features from notebook 03, including derived rate features.

In [ ]:
# Approach A: the three output percentages.
# This is what I naively tried first - it feels intuitive because these directly
# label how much of each archetype a player is. The problem is they sum to 1.0,
# which means I'm not giving K-Means three independent numbers; I'm giving it two.
PCT_FEATURES = ['pct_combat', 'pct_collect', 'pct_explore']

# Approach B: the normalized raw input features from notebook 03.
# I include the v2.2 derived rate features here because they're part of the
# production pipeline. Leaving them out would underrepresent what the real
# clustering actually uses.
RAW_FEATURES = [
    'enemiesHit', 'damageDone', 'timeInCombat', 'kills',
    'itemsCollected', 'pickupAttempts', 'timeNearInteractables',
    'distanceTraveled', 'timeSprinting',
    'damage_per_hit',       # v2.2 derived: damageDone / max(enemiesHit, 1)
    'pickup_attempt_rate',  # v2.2 derived: pickupAttempts / max(timeNearInteractables, 1)
]

# Hard stop if the pipeline wasn't run - better to fail loudly here than get
# a confusing KeyError somewhere in the middle of the experiment.
missing_pct = [f for f in PCT_FEATURES if f not in df_pct.columns]
missing_raw = [f for f in RAW_FEATURES if f not in df_raw.columns]

if missing_pct:
    raise KeyError(f"Missing percentage columns: {missing_pct}")
if missing_raw:
    raise KeyError(f"Missing raw columns: {missing_raw}")

# Fill NaN with equal split (1/3) for percentages - a session with zero activity
# shouldn't bias any single archetype.
X_pct = df_pct[PCT_FEATURES].fillna(1/3).values
X_raw = df_raw[RAW_FEATURES].fillna(0).values

print(f"Approach A (Percentages): {X_pct.shape[1]} features, {X_pct.shape[0]} rows")
print(f"Approach B (Raw):         {X_raw.shape[1]} features, {X_raw.shape[0]} rows")

## 3. Verify the Compositional Data Problem

Before clustering, I want to prove the constraint mathematically. If `pct_combat` and `pct_collect` are fixed, `pct_explore` is fully determined. This makes the 3D percentage space geometrically a 2D simplex - a triangle - not a cube.

In [ ]:
pct_df = df_pct[PCT_FEATURES].copy()

# The core claim: every row sums to exactly 1.0.
# If this holds, the third feature is always redundant.
print("=== Compositional Constraint Verification ===")
row_sums = pct_df.sum(axis=1)
print(f"Row sums - mean: {row_sums.mean():.6f}, std: {row_sums.std():.6f}, min: {row_sums.min():.6f}, max: {row_sums.max():.6f}")
print(f"All rows sum to 1.0: {np.allclose(row_sums, 1.0, atol=1e-6)}")

# The correlation matrix will show strong negative correlations between the
# percentage features. This is the mathematical fingerprint of the constraint -
# it's not a genuine relationship between behaviors, it's an artifact of the math.
print("\n=== Pearson Correlation Matrix (Percentage Features) ===")
print(pct_df.corr().round(4).to_string())

# For contrast, the raw feature correlations are driven by actual player behavior,
# not by a sum-to-one constraint.
print("\n=== Pearson Correlation Matrix (Raw Features, first 5) ===")
print(pd.DataFrame(X_raw, columns=RAW_FEATURES).iloc[:, :5].corr().round(4).to_string())

In [ ]:
# I want to show this geometrically, not just numerically.
# The left plot should look like a flat triangle hovering in 3D space - that's
# the simplex. The right plot should look like a proper 3D cloud.
# If these look similar, the experiment is broken.
fig = plt.figure(figsize=(14, 5))

# --- Percentage space ---
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X_pct[:, 0], X_pct[:, 1], X_pct[:, 2],
            alpha=0.3, s=4, c='steelblue')
ax1.set_xlabel('pct_combat')
ax1.set_ylabel('pct_collect')
ax1.set_zlabel('pct_explore')
ax1.set_title('Approach A: Percentage Space\n(data confined to a 2D simplex triangle)')
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1); ax1.set_zlim(0, 1)

# I use PCA to project the 11D raw space down to 3D for visualisation.
# This isn't the clustering space - it's just for the plot.
pca = PCA(n_components=3)
X_raw_pca = pca.fit_transform(X_raw)
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(X_raw_pca[:, 0], X_raw_pca[:, 1], X_raw_pca[:, 2],
            alpha=0.3, s=4, c='darkorange')
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
ax2.set_zlabel('PC3')
ax2.set_title('Approach B: Raw Feature Space (PCA)\n(data occupies full 3D volume)')

plt.tight_layout()
plt.savefig(os.path.join(EXP, 'pct_vs_raw_geometry.png'), dpi=120, bbox_inches='tight')
plt.show()
print("Geometry comparison saved to experiments/pct_vs_raw_geometry.png")

## 4. Run K-Means on Both Approaches

K-Means is run for K=2,3,4,5 on both feature sets. I record Silhouette, Davies-Bouldin, and Calinski-Harabasz for each.

In [ ]:
def run_kmeans(X, k, label):
    """
    I use n_init=10 here to reduce sensitivity to random centroid initialization.
    A single init can land in a local minimum, which would make one approach
    look worse than it really is and bias the comparison.
    """
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    n_actual = len(np.unique(labels))
    if n_actual < k:
        # This means K-Means collapsed two clusters into one.
        # It signals the feature space can't actually support k distinct groups.
        return None
    return {
        'approach':          label,
        'k':                 k,
        'silhouette':        round(silhouette_score(X, labels), 6),
        'davies_bouldin':    round(davies_bouldin_score(X, labels), 6),
        'calinski_harabasz': round(calinski_harabasz_score(X, labels), 2),
        'cluster_sizes':     np.bincount(labels).tolist(),
        'kmeans':            km,
        'labels':            labels,
    }

K_VALUES = [2, 3, 4, 5]
results = []
fitted = {}  # keep fitted objects so I can reuse labels for the shape plots later

for k in K_VALUES:
    r_pct = run_kmeans(X_pct, k, 'A_percentage')
    r_raw = run_kmeans(X_raw, k, 'B_raw')
    if r_pct:
        fitted[(k, 'pct')] = r_pct
        results.append({k2: v for k2, v in r_pct.items() if k2 not in ('kmeans', 'labels')})
    if r_raw:
        fitted[(k, 'raw')] = r_raw
        results.append({k2: v for k2, v in r_raw.items() if k2 not in ('kmeans', 'labels')})

results_df = pd.DataFrame(results)
print(results_df[['approach', 'k', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'cluster_sizes']].to_string(index=False))

## 5. Compare Metrics Side-by-Side

In [ ]:
# I plot all three metrics together so I can see if the percentage approach
# inflates Silhouette while also producing tighter (lower DB) clusters -
# which would be the signature of the simplex illusion rather than genuine separation.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

pct_rows = results_df[results_df['approach'] == 'A_percentage']
raw_rows = results_df[results_df['approach'] == 'B_raw']

metrics = [
    ('silhouette',        'Silhouette Score',        'Higher is better',  True),
    ('davies_bouldin',    'Davies-Bouldin Index',    'Lower is better',   False),
    ('calinski_harabasz', 'Calinski-Harabasz Score', 'Higher is better',  True),
]

for ax, (col, title, subtitle, higher_better) in zip(axes, metrics):
    ax.plot(pct_rows['k'], pct_rows[col], 'o--', color='steelblue',  label='A: Percentages', linewidth=2)
    ax.plot(raw_rows['k'], raw_rows[col], 's-',  color='darkorange', label='B: Raw',          linewidth=2)
    ax.set_xlabel('K (number of clusters)')
    ax.set_title(f"{title}\n({subtitle})")
    ax.set_xticks(K_VALUES)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Experiment C: Percentage vs Raw Feature Clustering', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(EXP, 'pct_vs_raw_metrics.png'), dpi=120, bbox_inches='tight')
plt.show()
print("Metric comparison saved to experiments/pct_vs_raw_metrics.png")

## 6. Cluster Shape Analysis at K=3

K=3 is the production-selected value. I want to see what the clusters actually look like geometrically under each approach - not just the scores.

In [ ]:
r3_pct = fitted.get((3, 'pct'))
r3_raw = fitted.get((3, 'raw'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#2ecc71', '#3498db']

# I plot pct_combat vs pct_explore for Approach A.
# The dashed diagonal is the sum=1 boundary - any point above it is impossible.
# If the cluster boundaries align with this line, it proves the clustering
# is responding to the math constraint rather than to actual player patterns.
ax = axes[0]
for cid in range(3):
    mask = r3_pct['labels'] == cid
    ax.scatter(X_pct[mask, 0], X_pct[mask, 2], alpha=0.3, s=5, color=colors[cid], label=f'Cluster {cid}')
ax.set_xlabel('pct_combat')
ax.set_ylabel('pct_explore')
ax.set_title('Approach A - Percentages (K=3)\nNote: axes are linearly dependent')
ax.legend(markerscale=3)
x_line = np.linspace(0, 1, 100)
ax.plot(x_line, 1 - x_line, 'k--', alpha=0.4, linewidth=1, label='sum=1 boundary')
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)

# For Approach B I project the 11D raw space to 2D via PCA.
# The clusters should look like natural blobs in a full space,
# not like they're squeezed onto a line.
ax = axes[1]
for cid in range(3):
    mask = r3_raw['labels'] == cid
    ax.scatter(X_raw_pca[mask, 0], X_raw_pca[mask, 1], alpha=0.3, s=5, color=colors[cid], label=f'Cluster {cid}')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Approach B - Raw Features PCA (K=3)\nAxes are independent')
ax.legend(markerscale=3)

plt.suptitle('Cluster Shape at K=3', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(EXP, 'pct_vs_raw_k3_clusters.png'), dpi=120, bbox_inches='tight')
plt.show()
print("K=3 cluster shapes saved to experiments/pct_vs_raw_k3_clusters.png")

## 7. Save Results

In [ ]:
out_path = os.path.join(EXP, 'pct_vs_raw_results.csv')
results_df.drop(columns=['cluster_sizes']).to_csv(out_path, index=False)
print(f"Results saved to {out_path}")

## 8. Findings

### Compositional Data Problem - Confirmed

The percentage features (`pct_combat + pct_collect + pct_explore = 1.0`) place all data on a **2D simplex** (triangle) inside 3D space. K-Means treats this triangle as if it were a full cube, producing clusters that are geometrically meaningful on the triangle but misleading in interpretation.

Key observations:

| Issue | Effect |
|---|---|
| Linear dependency between axes | One axis is always determinable from the other two - the 3rd dimension adds no information |
| Artificially inflated Silhouette | Clusters look well-separated on the simplex but that separation is geometrically forced |
| Cluster boundaries align with the sum=1 constraint | Boundaries reflect the mathematical constraint, not genuine player behaviour differences |
| No room for a "hybrid" player | A player high in all three scores is impossible - someone with high combat *must* have low explore |

### Why Raw Features Are Correct

Clustering on normalized raw features:
- Each axis is **independently measurable** (a player can travel far AND fight a lot)
- Captures the **causes** of playstyle, not the derived ratio symptoms
- Allows the K-Means Euclidean distance to function as designed
- Percentages are still computed *after* clustering (as soft membership scores via IDW) for interpretability

### Conclusion

The notebook `05_Clustering.ipynb` methodology note is validated. Clustering on percentages is geometrically invalid for K-Means. The production pipeline correctly clusters on raw normalized features (notebook 03 output) and derives percentages as a post-hoc interpretability layer.